In [0]:
print("spark version" + spark.version)

spark version4.2.0


In [0]:
import requests

url = 'https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M&community=RE&longitude=-38.51&latitude=-12.97&start=20250101&end=20250131&format=JSON'

resposta = requests.get(url)
dados = resposta.json()

temperaturas = dados["properties"]["parameter"]["T2M"]

df_bronze = spark.createDataFrame([
    {"data": dia, "temperatura_c": valor}
    for dia, valor in temperaturas.items()
])

display(df_bronze.orderBy("data"))


salvar_bronze = df_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "bronze_temperatura_mensal"
)

data,temperatura_c
20250101,28.18
20250102,28.12
20250103,28.35
20250104,28.58
20250105,28.99
20250106,28.77
20250107,28.5
20250108,28.47
20250109,29.07
20250110,26.54


In [0]:
from pyspark.sql.functions import year, month, dayofmonth, date_format
from pyspark.sql.functions import col, to_date
from pyspark.sql import functions as F

In [0]:
df_silver = df_bronze.withColumn(
    "data",
    F.to_date(F.col("data"), "yyyyMMdd")
)

display(df_silver)
df_silver.printSchema()


salvar_tabela = df_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver_temperatura_mensal"
)

data,temperatura_c
2025-01-01,28.18
2025-01-02,28.12
2025-01-03,28.35
2025-01-04,28.58
2025-01-05,28.99
2025-01-06,28.77
2025-01-07,28.5
2025-01-08,28.47
2025-01-09,29.07
2025-01-10,26.54


root
 |-- data: date (nullable = true)
 |-- temperatura_c: double (nullable = true)



In [0]:
gold_temperatura_mensal = (
    df_silver
    .groupBy(
        F.year("data").alias("ano"),
        F.month("data").alias("mes")
    )
    .agg(
        F.round(F.avg("temperatura_c"), 2).alias("temperatura_media_c"),
        F.round(F.max("temperatura_c"), 2).alias("maior_media_diaria_c"),
        F.round(F.min("temperatura_c"), 2).alias("menor_media_diaria_c"),
        F.count("temperatura_c").alias("dias_com_dados")
    )
)

display(gold_temperatura_mensal)

ano,mes,temperatura_media_c,maior_media_diaria_c,menor_media_diaria_c,dias_com_dados
2025,1,27.72,29.07,26.15,31


In [0]:
gold_temperatura_mensal.write.format("delta").mode("overwrite").saveAsTable(
    "gold_temperatura_mensal"
)